# Session 8 — Tool Use and Function Calling

LLMs are excellent at reasoning but cannot act on the world directly. Tool use (also called function calling) gives the model a way to request external actions — fetching live data, running calculations, searching a document store — and incorporate the results into its answer.

This notebook covers:
1. Anatomy of a tool definition (JSON schema)
2. A custom weather tool — the classic first example
3. The agentic loop step by step
4. Built-in tools: `web_search` and `code_interpreter`
5. Parallel tool calls — two tools in one turn
6. RAG as a tool call — the model decides when to retrieve

## Learning Goals

- understand the JSON schema format for tool definitions
- implement the four-step agentic loop (request → detect → execute → feed back)
- use built-in OpenAI tools without writing a Python function
- define multiple tools and handle parallel calls
- refactor the RAG pipeline from notebook 04 so the model controls retrieval

In [1]:
import json
import os
import requests
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_ORG_ID = os.getenv("OPENAI_ORG_ID")
OPENAI_PROJECT_ID = os.getenv("OPENAI_PROJECT_ID")

print("OpenAI key present:", bool(OPENAI_API_KEY))


def make_openai_client(api_key=None, base_url=None):
    kwargs = {}
    if api_key:
        kwargs["api_key"] = api_key
    if base_url:
        kwargs["base_url"] = base_url
    if api_key == OPENAI_API_KEY and not base_url:
        if OPENAI_ORG_ID:
            kwargs["organization"] = OPENAI_ORG_ID
        if OPENAI_PROJECT_ID:
            kwargs["project"] = OPENAI_PROJECT_ID
    return OpenAI(**kwargs)


def require_env(name: str):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


def session8_dir() -> Path:
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    candidate = cwd / "material" / "Session 8"
    if candidate.exists():
        return candidate
    return cwd

OpenAI key present: True


## 1. Anatomy of a Tool Definition

Every tool is a Python dict that follows the OpenAI JSON schema format. The model reads `description` to decide when to call the tool, and `parameters` to know what arguments to pass.

In [2]:
# Annotated tool schema — nothing executes here, just for reading

weather_tool = {
    "type": "function",           # always "function" for custom tools
    "name": "get_weather",        # must match the Python function name you will call
    "description": (
        "Returns current weather conditions for a city. "
        "Call this whenever the user asks about weather or temperature."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. Sydney, Paris, New York",
            },
            "units": {
                "type": "string",
                "enum": ["celsius", "fahrenheit"],
                "description": "Temperature unit to return.",
            },
        },
        "required": ["location", "units"],
        "additionalProperties": False,
    },
    "strict": True,   # model must fill all required fields; no extra fields allowed
}

print(json.dumps(weather_tool, indent=2))

{
  "type": "function",
  "name": "get_weather",
  "description": "Returns current weather conditions for a city. Call this whenever the user asks about weather or temperature.",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {
        "type": "string",
        "description": "City name, e.g. Sydney, Paris, New York"
      },
      "units": {
        "type": "string",
        "enum": [
          "celsius",
          "fahrenheit"
        ],
        "description": "Temperature unit to return."
      }
    },
    "required": [
      "location",
      "units"
    ],
    "additionalProperties": false
  },
  "strict": true
}


## 2. Custom Tool: Weather

`get_weather` calls the free wttr.in API — no API key needed. The function signature must match the `name` and `parameters` in the schema above.

In [3]:
def get_weather(location: str, units: str = "celsius") -> dict:
    """Fetch current weather from wttr.in (free, no API key required)."""
    unit_char = "m" if units == "celsius" else "u"
    url = f"https://wttr.in/{requests.utils.quote(location)}?format=j1&{unit_char}"
    try:
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        current = data["current_condition"][0]
        temp = current["temp_C"] if units == "celsius" else current["temp_F"]
        desc = current["weatherDesc"][0]["value"]
        return {
            "location": location,
            "temperature": int(temp),
            "units": units,
            "description": desc,
        }
    except Exception as e:
        return {"error": str(e), "location": location}


# Quick smoke test — confirm the function works before wiring it to the model
result = get_weather("Sydney", "celsius")
print(result)

{'location': 'Sydney', 'temperature': 28, 'units': 'celsius', 'description': 'Sunny'}


## 3. The Agentic Loop

Tool use works in four steps:

1. **Request** — send the user message plus the tool list to the model
2. **Detect** — check if the model returned a `function_call` in `response.output`
3. **Execute** — call the matching Python function with the model's arguments
4. **Feed back** — append the result as `function_call_output` and call the model again

The model then produces a final answer that incorporates the tool result.

### Step 1 — Send the request with tools

In [4]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

input_messages = [
    {"role": "user", "content": "What is the weather like in Tokyo right now? Use celsius."}
]

response = client.responses.create(
    model="gpt-4o-mini",
    tools=[weather_tool],
    input=input_messages,
    tool_choice="auto",   # model decides whether to call a tool
)

print("Output types:", [item.type for item in response.output])

Output types: ['function_call']


### Step 2 — Detect the function call

The model signals it wants to call a tool by including a `function_call` item in `response.output`. We inspect the output list and find it.

In [5]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

tool_call = None
for item in response.output:
    if item.type == "function_call":
        tool_call = item
        break

if tool_call:
    print("Function to call:", tool_call.name)
    print("Arguments (JSON string):", tool_call.arguments)
    print("Call ID:", tool_call.call_id)
else:
    print("Model did not request a tool call.")

Function to call: get_weather
Arguments (JSON string): {"location":"Tokyo","units":"celsius"}
Call ID: call_yHM5jom2Se761B5yCTufdK3B


### Step 3 — Execute the function locally

We parse the JSON arguments and call the matching Python function.

In [6]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

if tool_call:
    args = json.loads(tool_call.arguments)
    print("Parsed arguments:", args)

    # Dispatch: call the function whose name matches tool_call.name
    tool_result = get_weather(**args)
    print("Tool result:", tool_result)

Parsed arguments: {'location': 'Tokyo', 'units': 'celsius'}
Tool result: {'location': 'Tokyo', 'temperature': 18, 'units': 'celsius', 'description': 'Partly cloudy'}


### Step 4 — Feed the result back and get the final answer

We append the model's tool call output and a `function_call_output` message, then call the model again.

In [7]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

if tool_call:
    # Append the model's tool call items to the conversation
    input_messages = input_messages + list(response.output)

    # Append our tool result
    input_messages.append({
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(tool_result),
    })

    # Second model call — now it has the tool result
    final_response = client.responses.create(
        model="gpt-4o-mini",
        tools=[weather_tool],
        input=input_messages,
        tool_choice="auto",
    )

    display(Markdown(final_response.output_text))

The current weather in Tokyo is 18°C and partly cloudy.

### Clean Helper: `run_tool_loop`

The four steps above always follow the same pattern. We can wrap them in a reusable function.

In [ ]:
# Map of tool name -> callable Python function
TOOL_REGISTRY = {
    "get_weather": get_weather,
}


def run_tool_loop(
    user_message: str,
    tools: list,
    registry: dict,
    model: str = "gpt-4o-mini",
    max_rounds: int = 5,
) -> str:
    """
    Run the agentic tool loop until the model stops calling tools or max_rounds is reached.
    Returns the final text response.
    """
    client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))
    messages = [{"role": "user", "content": user_message}]

    for _ in range(max_rounds):
        response = client.responses.create(
            model=model,
            tools=tools,
            input=messages,
            tool_choice="auto",
        )

        calls = [item for item in response.output if item.type == "function_call"]

        if not calls:
            return response.output_text

        messages = messages + list(response.output)

        for call in calls:
            fn = registry.get(call.name)
            if fn is None:
                result = {"error": f"Unknown tool: {call.name}"}
            else:
                print(f"Calling tool '{call.name}' with arguments: {call.arguments}")
                result = fn(**json.loads(call.arguments))

            messages.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result),
            })

    return "Max tool rounds reached without a final answer."

In [11]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

answer = run_tool_loop(
    user_message="What is the weather in London and Paris? Give me celsius for both.",
    tools=[weather_tool],
    registry=TOOL_REGISTRY,
)
display(Markdown(answer))

Calling tool 'get_weather' with arguments: {"location":"London","units":"celsius"}
Calling tool 'get_weather' with arguments: {"location":"Paris","units":"celsius"}


The current weather is as follows:

- **London**: 11°C, Sunny
- **Paris**: 10°C, Clear

## 4. Built-in Tools

OpenAI provides built-in tools you can enable with a single dict — no Python function required. The model calls them server-side and returns the result inline.

The two most useful for teaching:
- `web_search` — live web retrieval
- `code_interpreter` — the model writes and executes Python in a sandbox

### `web_search`

Just declare `{"type": "web_search"}` in the tools list. No schema needed, no Python function to write.

In [12]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

web_response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{"type": "web_search"}],
    input="What were the main AI announcements from the last week?",
)

display(Markdown(web_response.output_text))

Over the past week, several significant developments have emerged in the AI sector:

**Enterprise AI Integration**

Enterprise AI is transitioning from experimental tools to trusted digital coworkers embedded in daily business operations. Analysts forecast that nearly half of enterprise applications will incorporate task-specific AI agents within the next year, driven by advancements in contextual memory, local on-device processing, and workflow automation. This shift aims to eliminate operational inefficiencies by automating mundane tasks and enhancing contextual understanding, enabling agents to act intelligently like trained employees. However, challenges remain, including trust issues and the need for robust security measures. ([techradar.com](https://www.techradar.com/pro/2026-the-year-enterprise-ai-finally-gets-to-work?utm_source=openai))

**Anthropic's Infrastructure Expansion**

Anthropic has secured multi-gigawatt compute capacity with Google and Broadcom, enabling the training of larger models and more efficient inference processes. This expansion is expected to enhance the capabilities of their AI systems, including Claude Code, and support the development of new models like Claude Mythos Preview, designed for defensive cybersecurity applications. ([fordelstudios.com](https://fordelstudios.com/research/ai-race-scorecard-week-april-7-2026?utm_source=openai))

**OpenAI's Codex Pricing Update**

OpenAI has transitioned Codex, its AI coding tool, to a pay-as-you-go model priced at $20 per seat. This move aims to make the tool more accessible to a broader range of developers and organizations, potentially increasing its adoption and integration into various development workflows. ([fordelstudios.com](https://fordelstudios.com/research/ai-race-scorecard-week-april-7-2026?utm_source=openai))

**Cursor 3 Release**

Cursor has launched Cursor 3, a redesigned agent-first coding platform that directly competes with Anthropic's Claude Code and OpenAI's Codex. Key features include an Agents Window for running multiple AI agents in parallel, a Design Mode for annotating UI elements in a browser, and Cloud Agents for offloading heavy tasks to Cursor's servers for faster execution. ([rawpickai.com](https://rawpickai.com/blog/ai-tool-news-roundup-april-7-2026?utm_source=openai))

**Regulatory Developments**

Washington State has enacted new laws mandating AI labels on altered media and restricting chatbot interactions with minors. These regulations aim to increase transparency and protect vulnerable populations from potential misuse of AI technologies. ([creati.ai](https://creati.ai/ai-news/2026-04-05/?utm_source=openai))


## Highlights:
- [2026: The year enterprise AI finally gets to work](https://www.techradar.com/pro/2026-the-year-enterprise-ai-finally-gets-to-work?utm_source=openai), Published on Friday, April 03 

### `code_interpreter`

The model writes Python, executes it in a sandboxed environment, and returns the result. Useful for maths, data transformation, or anything a small script can solve.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

from IPython.display import Image as IPImage

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

code_response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
    input="Generate a random distribution of 100 numbers and plot a histogram with matplotlib.",
)

# Show what item types came back — you should see 'code_interpreter_call' and 'message'
print("Output item types:", [item.type for item in code_response.output])

# Show the code the model wrote and ran
for item in code_response.output:
    if item.type == "code_interpreter_call":
        print("\n--- Code the model wrote and ran ---")
        print(item.code)

# The plot is returned as a container_file_citation annotation inside the message.
# Retrieve the PNG bytes directly from the container files API and display inline.
api_key = require_env("OPENAI_API_KEY")
for item in code_response.output:
    if item.type == "message":
        for block in item.content:
            for annotation in getattr(block, "annotations", []):
                if annotation.type == "container_file_citation":
                    print(f"\nRetrieving plot: {annotation.file_id}")
                    file_resp = requests.get(
                        f"https://api.openai.com/v1/containers/{annotation.container_id}"
                        f"/files/{annotation.file_id}/content",
                        headers={"Authorization": f"Bearer {api_key}"},
                        timeout=30,
                    )
                    file_resp.raise_for_status()
                    display(IPImage(data=file_resp.content))

display(Markdown(code_response.output_text))

In [25]:
print(code_response.model_dump_json(indent=2))

{
  "id": "resp_0bdd21c751c8c50b0069d74cb8ee8881959c7b3efd72145f16",
  "created_at": 1775717560.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4o-mini-2024-07-18",
  "object": "response",
  "output": [
    {
      "id": "ci_0bdd21c751c8c50b0069d74cb9c0d08195a6617a9b9a564633",
      "code": "import numpy as np\nimport matplotlib.pyplot as plt\n\n# Generate a random distribution of 100 numbers\nrandom_numbers = np.random.rand(100)\n\n# Plot the histogram\nplt.figure(figsize=(10, 6))\nplt.hist(random_numbers, bins=10, color='skyblue', edgecolor='black')\nplt.title('Histogram of Random Distribution')\nplt.xlabel('Value')\nplt.ylabel('Frequency')\nplt.grid(axis='y', alpha=0.75)\nplt.show()",
      "container_id": "cntr_69d74cb94a5c81908a2a1a5b0f0c266b011f2498e21715f4",
      "outputs": null,
      "status": "completed",
      "type": "code_interpreter_call"
    },
    {
      "id": "msg_0bdd21c751c8c50b0069d74cca016c8195901636b20f

### Key Difference from Custom Tools

| | Custom tool | Built-in tool |
|---|---|---|
| Python function needed? | Yes | No |
| Runs where? | Your machine | OpenAI servers |
| Can access internet? | If your function does | Yes (`web_search`) |
| Can run code? | If your function does | Yes (`code_interpreter`) |

## 5. Parallel Tool Calls

When multiple tools are registered, the model may call several in a single turn. This is called parallel tool calling and it saves a round-trip for questions that need two independent lookups.

In [ ]:
def get_current_date() -> dict:
    """Return today's date — no external API needed."""
    from datetime import date
    today = date.today()
    return {
        "date": today.isoformat(),
        "day_of_week": today.strftime("%A"),
    }


date_tool = {
    "type": "function",
    "name": "get_current_date",
    "description": "Returns today's date and day of the week. Call this when the user asks what day or date it is.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
    "strict": True,
}

# Smoke test
print(get_current_date())

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

TOOL_REGISTRY_V2 = {
    "get_weather": get_weather,
    "get_current_date": get_current_date,
}

answer = run_tool_loop(
    user_message=(
        "What day is it today, and what is the weather in Melbourne right now in celsius?"
    ),
    tools=[weather_tool, date_tool],
    registry=TOOL_REGISTRY_V2,
)

display(Markdown(answer))

The model may issue both `get_weather` and `get_current_date` calls in the same response turn. The `run_tool_loop` helper handles this naturally because it loops over all `function_call` items before making the next model call.

## 6. RAG as a Tool Call

In notebook 04, context is always injected into the prompt — the model never chooses to retrieve. That works but has two costs:
- Every query pays the embedding + search cost, even when no retrieval is needed
- The model cannot say "I already know this, I do not need to look it up"

Wrapping the FAISS search as a tool lets the model decide when retrieval adds value.

In [ ]:
import sys
import numpy as np

sys.path.insert(0, str(session8_dir()))

from helpers.pdf_utils import extract_pdf_text, join_pages
from helpers.rag_utils import chunk_text, package_chunks, build_faiss_index, search_index

# Requires OPENAI_API_KEY — embeddings are needed to build the index.
# Skip this cell if you do not have live API access.

pdf_dir = session8_dir() / "data" / "pdfs"
pdf_paths = sorted(pdf_dir.glob("*.pdf"))
print(f"PDFs found: {[p.name for p in pdf_paths]}")

# Load and chunk all PDFs
all_chunks = []
for pdf_path in pdf_paths:
    pages = extract_pdf_text(pdf_path)
    full_text = join_pages(pages)
    chunks = chunk_text(full_text, chunk_size=500, overlap=100)
    all_chunks.extend(package_chunks(chunks, pdf_path.name))

print(f"Total chunks: {len(all_chunks)}")

# Embed all chunks with OpenAI
client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))
chunk_texts_list = [c["text"] for c in all_chunks]
emb_response = client.embeddings.create(model="text-embedding-3-small", input=chunk_texts_list)
chunk_embeddings = np.array([item.embedding for item in emb_response.data], dtype="float32")

# Build FAISS index
faiss_index = build_faiss_index(chunk_embeddings)
print(f"FAISS index ready. Vectors: {faiss_index.ntotal}")

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

def search_hr_docs(query: str, top_k: int = 4) -> str:
    """
    Search the HR document knowledge base (FAISS index built above).
    Returns the top_k most relevant text chunks joined as a single string.
    """
    client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))
    query_embedding = np.array(
        client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding,
        dtype="float32",
    )
    distances, indices = search_index(faiss_index, query_embedding, top_k=top_k)
    retrieved = [all_chunks[idx]["text"] for idx in indices[0]]
    return "

---

".join(retrieved)


search_tool = {
    "type": "function",
    "name": "search_hr_docs",
    "description": (
        "Search the HR document knowledge base for information about employee benefits, "
        "health plans, dental coverage, PTO policy, and similar HR topics. "
        "Call this tool when the user asks a question that requires specific policy details."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query derived from the user's question.",
            },
            "top_k": {
                "type": "integer",
                "description": "Number of document chunks to retrieve. Default is 4.",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    "strict": True,
}

# Smoke test — confirm search works before wiring to the model
snippet = search_hr_docs("dental coverage annual limit")
print(snippet[:300])

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

TOOL_REGISTRY_RAG = {
    "search_hr_docs": search_hr_docs,
}

# Question that needs retrieval
answer_retrieval = run_tool_loop(
    user_message="What does the employee handbook say about performance reviews?",
    tools=[search_tool],
    registry=TOOL_REGISTRY_RAG,
)
display(Markdown("### Answer with retrieval
" + answer_retrieval))

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

# Question that does NOT need retrieval — model should answer from general knowledge
answer_no_retrieval = run_tool_loop(
    user_message="What does the acronym RAG stand for in machine learning?",
    tools=[search_tool],
    registry=TOOL_REGISTRY_RAG,
)
display(Markdown("### Answer without retrieval
" + answer_no_retrieval))

Notice that the model calls `search_hr_docs` for the HR policy question but answers directly for the RAG acronym. The model reasons about when retrieval is needed.

### Why this matters

| Pattern | Who controls retrieval? | Cost |
|---|---|---|
| Notebook 04 (always inject) | Developer (always) | Always pays embedding + search |
| Tool call (this notebook) | Model (as needed) | Pays only when retrieval helps |

For production RAG apps, combining both patterns is common: tool calling for the main flow, with injected context for critical baseline information.

## Recap

- Tools are JSON schemas that tell the model what functions exist and when to call them.
- The agentic loop has four steps: request → detect → execute → feed back.
- `run_tool_loop` encapsulates the loop for reuse.
- Built-in tools (`web_search`, `code_interpreter`) need no Python function.
- The model handles parallel calls naturally when multiple tools are registered.
- RAG-as-a-tool lets the model decide when to retrieve, reducing unnecessary embedding cost.

**Next:** `06_streamlit_rag_app.py` combines RAG, streaming, and a Streamlit UI into a complete deployable app.